# Lab 3A: Tools and structured output
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 3 · about 30 minutes**

**Objectives**
- Run the tool use loop and return results correctly
- Handle parallel tool calls and tool errors
- Compare tool_choice settings
- Get schema-valid JSON and validate its meaning

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

In [ ]:
# Synthetic ticket system (training data only)
TICKETS = {
    "T-77": {"ticket_id": "T-77", "customer": "C-1001", "status": "open", "summary": "Router drops every evening since 20 Sep"},
    "T-78": {"ticket_id": "T-78", "customer": "C-1002", "status": "closed", "summary": "Billing address change"},
    "T-81": {"ticket_id": "T-81", "customer": "C-1002", "status": "open", "summary": "Router reboot loop after firmware update"},
}
def get_ticket(ticket_id):
    t = TICKETS.get(ticket_id.strip().upper())
    if t is None:
        raise KeyError(f"No ticket {ticket_id}. Ticket IDs look like T-77.")
    return t
def update_ticket_status(ticket_id, status):
    t = get_ticket(ticket_id); t["status"] = status
    return {"ticket_id": t["ticket_id"], "status": status}

TOOLS = [
    {"name": "get_ticket", "description": "Look up ONE support ticket by ID (format T-77). Returns customer, status, and summary.",
     "input_schema": {"type": "object", "properties": {"ticket_id": {"type": "string"}}, "required": ["ticket_id"]}},
    {"name": "update_ticket_status", "description": "Set a ticket's status. Only use when the user explicitly asks to change status.",
     "input_schema": {"type": "object", "properties": {"ticket_id": {"type": "string"},
                      "status": {"type": "string", "enum": ["open", "pending", "closed"]}}, "required": ["ticket_id", "status"]}},
]
FUNCS = {"get_ticket": get_ticket, "update_ticket_status": update_ticket_status}
def execute(name, args):
    try:
        return json.dumps(FUNCS[name](**args)), False
    except Exception as e:
        return json.dumps({"error": str(e)}), True

## Part 1: the tool use loop
Claude asks; your code runs the tool and returns a `tool_result` with the matching `tool_use_id`.

In [ ]:
def run_with_tools(user_text, tools=TOOLS, max_rounds=8, verbose=True, **kw):
    messages = [{"role": "user", "content": user_text}]
    for _ in range(max_rounds):                      # safety cap, not the stopping rule
        resp = client.messages.create(model=config.MODEL, max_tokens=config.MAX_TOKENS, tools=tools, messages=messages, **kw)
        if verbose: log_response(resp)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return resp, messages
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                content, is_error = execute(b.name, b.input)
                if verbose: print(f"    ran {b.name}({b.input}) -> is_error={is_error}")
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": content, "is_error": is_error})
        messages.append({"role": "user", "content": results})    # every tool_use answered, in one message
        kw.pop("tool_choice", None)                              # forced choice applies to the first turn only
    raise RuntimeError("Safety cap reached")

resp, _ = run_with_tools("What's the status of ticket T-77, and what is it about?")

## Part 2: parallel calls and errors
**Predict:** how many `tool_use` blocks arrive in the first response? What happens with the unknown ticket?

In [ ]:
resp, msgs = run_with_tools("Compare tickets T-77, T-81 and T-99: which are still open?")
first = msgs[1]["content"]
print("\ntool_use blocks in first reply:", sum(1 for b in first if b.type == "tool_use"))

T-99 does not exist, so its result came back with `is_error: true` and Claude explained it instead of crashing. Now see what the API says when a tool call goes unanswered:

In [ ]:
r = client.messages.create(model=config.MODEL, max_tokens=500, tools=TOOLS,
                           messages=[{"role": "user", "content": "Look up T-77 and T-81."}])
uses = [b for b in r.content if b.type == "tool_use"]
if len(uses) >= 2:
    partial = [{"role": "user", "content": "Look up T-77 and T-81."}, {"role": "assistant", "content": r.content},
               {"role": "user", "content": [{"type": "tool_result", "tool_use_id": uses[0].id, "content": "ok"}]}]
    try:
        client.messages.create(model=config.MODEL, max_tokens=200, tools=TOOLS, messages=partial)
    except anthropic.BadRequestError as e:
        print("400 as expected:", str(e)[:200])
else:
    print("Claude made", len(uses), "call(s) this time; rerun to see parallel calls.")

## Part 3: tool_choice
**Predict** the result for each setting on a greeting that needs no tool.

In [ ]:
for choice in [{"type": "auto"}, {"type": "any"}, {"type": "tool", "name": "get_ticket"}, {"type": "none"}]:
    try:
        r = client.messages.create(model=config.MODEL, max_tokens=300, tools=TOOLS, tool_choice=choice,
                                   messages=[{"role": "user", "content": "Hi! Just saying thanks for fixing my router."}])
        log_response(r, f"[{choice['type']}]")
    except anthropic.BadRequestError as e:
        print(f"[{choice['type']}] rejected by this model: {str(e)[:120]}")

## Part 4: JSON outputs with `output_config.format`

In [ ]:
TRIAGE_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["billing", "outage", "device", "account", "other"]},
        "priority": {"type": "string", "enum": ["P1", "P2", "P3"]},
        "summary": {"type": "string"},
        "customer_id": {"type": ["string", "null"]},
    },
    "required": ["category", "priority", "summary", "customer_id"],
    "additionalProperties": False,
}
INBOX = [
    "Whole street in sector 4 has no internet since 6am. Hospital nearby affected!",
    "Hi, account C-1002 here, I was charged twice for September.",
    "My router's WiFi light blinks orange. Tried restarting.",
    "How do I change the email on my account?",
    "Your app keeps logging me out.",
]
SYSTEM = "Triage telecom support messages. P1 = service outage affecting many users or critical services; P2 = one customer blocked; P3 = everything else."
triaged = []
for msg in INBOX:
    r = client.messages.create(model=config.MODEL, max_tokens=400, system=SYSTEM,
                               output_config={"format": {"type": "json_schema", "schema": TRIAGE_SCHEMA}},
                               messages=[{"role": "user", "content": msg}])
    data = json.loads(text_of(r)); triaged.append((msg, data))
    print(data)

Compare with asking for JSON in the prompt only. Count how many replies parse cleanly.

In [ ]:
ok = 0
for msg in INBOX:
    r = client.messages.create(model=config.MODEL, max_tokens=400, system=SYSTEM + " Reply in JSON with category, priority, summary, customer_id.",
                               messages=[{"role": "user", "content": msg}])
    try:
        json.loads(text_of(r)); ok += 1
    except json.JSONDecodeError:
        print("  did not parse:", text_of(r)[:80].replace("\n", " "), "...")
print(f"prompt-only JSON parsed cleanly: {ok}/{len(INBOX)}")

## Part 5: strict tool use
`"strict": true` guarantees tool inputs match `input_schema`.

In [ ]:
strict_tools = [dict(TOOLS[1], strict=True, input_schema={**TOOLS[1]["input_schema"], "additionalProperties": False})]
r = client.messages.create(model=config.MODEL, max_tokens=300, tools=strict_tools,
                           tool_choice={"type": "tool", "name": "update_ticket_status"},
                           messages=[{"role": "user", "content": "Please mark T-81 as waiting on the customer."}])
call = next(b for b in r.content if b.type == "tool_use")
print("tool input:", call.input, "| status is a valid enum value:", call.input["status"] in ["open", "pending", "closed"])

## Part 6: schema-valid is not the same as correct
Check a business rule in code: an outage that affects many users must be P1.

In [ ]:
def rule_violations(msg, d):
    issues = []
    if d["category"] == "outage" and re.search(r"street|sector|hospital|everyone|whole", msg, re.I) and d["priority"] != "P1":
        issues.append("wide outage not marked P1")
    if d["customer_id"] and not re.fullmatch(r"C-\d{4}", d["customer_id"]):
        issues.append(f"bad customer_id format: {d['customer_id']}")
    if d["customer_id"] and d["customer_id"] not in msg:
        issues.append("customer_id not present in the message (possible invention)")
    return issues
for msg, d in triaged:
    print(f"{d['priority']} {d['category']:8s} issues={rule_violations(msg, d) or 'none'}")

## You should now have
- [ ] `run_with_tools()` answering every tool call, with is_error on failures
- [ ] A tool_choice comparison
- [ ] Triage JSON via `output_config.format`
- [ ] Business-rule checks on schema-valid output

## Check yourself
1. What must every tool_use block receive before the next request?
2. When would you use strict tool use instead of JSON outputs?
3. What does structured output NOT guarantee?